In [ ]:
import pyemma
from pyemma.util.contexts import settings
import pyemma.msm as msm
import pyemma.plots as mplt
import mdtraj as md
import pickle
import matplotlib.pyplot as plt
import numpy as np
import nglview as nv
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
from IPython.display import display
import sys
sys.path.append("../")
from descriptors import *

In [ ]:
def get_torsions_in_traj(traj, mol):
    torsion_ring = get_torsions_idx_mol(mol)[1]
    torsion_non_ring = get_torsions_idx_mol(mol)[0]
    torsion_all = torsion_ring + torsion_non_ring
    torsions = md.compute_dihedrals(traj, torsion_all)  # shape (n_frames, n_torsions)
    return torsions

def get_ref_and_generated_traj(molecule_prefix, plot_distr=False, plot_structure=False):
    '''
    Given a molecule prefix, load the corresponding trajectory,
    extract the dihedral angles, and return the torsion trajectory object.
    We have 5 independent trajs for each molecules.
    Use the first four as ref, and the last one as generated.
    '''
    prefix_idx = molecule_prefix.split('_')[-1]
    molecule_prefix = molecule_prefix[:-len(prefix_idx)]
    ref_xtc = [f'eval/data_sample/{molecule_prefix}{str(i)}\\traj.xtc' for i in range(int(prefix_idx),int(prefix_idx)+4)]
    generated_xtc = f'eval/data_sample/{molecule_prefix}{str(int(prefix_idx)+4)}\\traj.xtc'
    pdb_path = f'eval/data_sample/{molecule_prefix}{str(int(prefix_idx)+4)}\\system.pdb'
    mol_path = f'eval/data_sample/{molecule_prefix}{str(int(prefix_idx)+4)}\\mol.pkl'
    mol = pickle.load(open(mol_path, 'rb'))
    # get torsion atom indices
    torsion_index = np.array(get_torsions_idx_mol(mol)[1]+get_torsions_idx_mol(mol)[0]).reshape(-1,4)
    feat_dihedral = pyemma.coordinates.featurizer(pdb_path)
    # Use cossin=True when we want to represent dihedral angles as their sine and cosine components.
    # This is useful for avoiding discontinuities at the 0/2π boundary.
    feat_dihedral.add_dihedrals(torsion_index, cossin=True, periodic=False)
    num_dihedrals = feat_dihedral.dimension() // 2  # Each dihedral is represented by two features (cos and sin)
    print(f'there are {num_dihedrals} dihedrals in the system')
    ref_traj = pyemma.coordinates.load(ref_xtc, features=feat_dihedral)
    generated_traj = pyemma.coordinates.load([generated_xtc], features=feat_dihedral)
    if plot_distr:
        # plot the distribution of dihedrals
        fig, ax = plt.subplots(figsize=(12, 8))
        pyemma.plots.plot_feature_histograms(np.concatenate(ref_traj), feature_labels=feat_dihedral, ax=ax)
        fig.tight_layout()
    if plot_structure:
        # Create a 2D visualization with RDKit
        mol_copy = Chem.Mol(mol)
        AllChem.Compute2DCoords(mol_copy)

        # Create highlighted depictions for each dihedral
        fig = plt.figure(figsize=(12, 8))
        for i, indices in enumerate(torsion_index):
            plt.subplot(4, 3, i+1)
            atoms = [int(idx) for idx in indices]
            img = Draw.MolToImage(mol_copy, highlightAtoms=atoms, highlightBonds=[])
            plt.imshow(img)
            plt.title(f"Dihedral {i+1}: {'-'.join(map(str, atoms))}")
            plt.axis('off')
        plt.tight_layout()
        # Plot dihedral angles over time for the ref trajectories
        plt.figure(figsize=(10, 6))
        for i in range(num_dihedrals):
            plt.plot(np.concatenate(ref_traj)[:, i], label=f'Dihedral {i+1}', alpha=0.5)
        plt.xlabel('Frame')
        plt.ylabel('Dihedral Angle (radians)')
        plt.title('Dihedral Angles Over Time')
        plt.legend()
        plt.tight_layout()
    return ref_traj, generated_traj, num_dihedrals

In [ ]:
print(ref_traj[0].shape, generated_traj.shape)

In [ ]:
ref_traj = [traj[::20] for traj in ref_traj]
generated_traj = generated_traj[::20]

In [ ]:
ref_molecule_prefix = 'qm_0_results\CCCC_C_H_1C_CCC1_25'
ref_traj, generated_traj, num_dihedrals = get_ref_and_generated_traj(ref_molecule_prefix, plot_distr=True, plot_structure=True)

In [ ]:
# molecule_prefix = 'qm_1_results\\C_C_H_1C_C_H_CCCO_C1_'
# molecule_prefix = 'qm_0_results\\CC_C_C_H_C_C_H_O_CO_'
# molecule_prefix = 'qm_0_results\\CC_CCC_C_H_O_CC_'
# molecule_prefix = 'qm_0_results\\CCCC_C_H_1C_CCC1_'
# molecule_prefix = "qm_0_results\\CCCC_CCC_C_C_"
# # molecule_prefix = '0_results\\Cc1ccccc1NC_O_c1ccc2c_c1_OCCO2_'
# # molecule_prefix = '7_results\O_C_NNC_O_c1cc2ccccc2cc1O_Nc1ccc_F_cc1F_'
# start_idx = 30

# xtc_path = f'eval/data_sample/{molecule_prefix}{str(start_idx)}\\traj.xtc'
# pdb_path = f'eval/data_sample/{molecule_prefix}{str(start_idx)}\\system.pdb'
# mol_path = f'eval/data_sample/{molecule_prefix}{str(start_idx)}\\mol.pkl'

# trajs = [f"eval/data_sample/{molecule_prefix}{str(i)}\\traj.xtc" for i in range(start_idx,start_idx+1)]
# traj = md.load(xtc_path, top=pdb_path)
# generated_traj = md.load(f"eval/data_sample/{molecule_prefix}{str(start_idx+1)}\\traj.xtc", top=f"eval/data_sample/{molecule_prefix}{str(start_idx+1)}\\system.pdb")
# mol = pickle.load(open(mol_path, 'rb'))

In [ ]:
# # get torsion atom indices
# torsion_index = np.array(get_torsions_idx_mol(mol)[1]+get_torsions_idx_mol(mol)[0]).reshape(-1,4)
# print(torsion_index)

# feat_dihedral = pyemma.coordinates.featurizer(pdb_path)
# # Use cossin=True when we want to represent dihedral angles as their sine and cosine components.
# # This is useful for avoiding discontinuities at the 0/2π boundary.
# feat_dihedral.add_dihedrals(torsion_index, cossin=True, periodic=False)
# print(feat_dihedral.dimension())

# dihedral_data = pyemma.coordinates.load(trajs, features=feat_dihedral)

# generated_feat_dihedral = pyemma.coordinates.featurizer(f"eval/data_sample/{molecule_prefix}{str(start_idx+1)}\\system.pdb")
# generated_feat_dihedral.add_dihedrals(torsion_index, cossin=True, periodic=False)

# generated_dihedral_data = pyemma.coordinates.load([f"eval/data_sample/{molecule_prefix}{str(start_idx+1)}\\traj.xtc"], features=generated_feat_dihedral)

# # Plot distribution of values in one feature
# fig, ax = plt.subplots(figsize=(12, 8))
# pyemma.plots.plot_feature_histograms(dihedral_data, feature_labels=feat_dihedral, ax=ax)
# fig.tight_layout()

In [ ]:
# ########## Visualization of dihedrals and structures ##########


# import matplotlib.pyplot as plt

# # # Get dihedral angles from the trajectory
# # dihedral_angles = get_torsions_in_traj(traj, mol)

# # # Create a 3D visualization of the molecule with NGLview
# # view = nv.show_mdtraj(traj[0])
# # view.clear_representations()
# # view.add_representation('licorice', selection='all')
# # view.center()

# # # Add representations to highlight atoms involved in dihedrals
# # colors = ['red', 'orange', 'yellow', 'green', 'blue', 'purple']
# # for i, indices in enumerate(torsion_index):
# #     color = colors[i % len(colors)]
# #     view.add_representation('ball+stick', selection=f'@{indices[0]} @{indices[1]} @{indices[2]} @{indices[3]}', color=color)

# # Create a 2D visualization with RDKit
# mol_copy = Chem.Mol(mol)
# AllChem.Compute2DCoords(mol_copy)

# # Create highlighted depictions for each dihedral
# fig = plt.figure(figsize=(12, 8))
# for i, indices in enumerate(torsion_index):
#     plt.subplot(4, 3, i+1)
#     atoms = [int(idx) for idx in indices]
#     img = Draw.MolToImage(mol_copy, highlightAtoms=atoms, highlightBonds=[])
#     plt.imshow(img)
#     plt.title(f"Dihedral {i+1}: {'-'.join(map(str, atoms))}")
#     plt.axis('off')

# plt.tight_layout()

# # Plot dihedral angles over time for the first trajectory
# plt.figure(figsize=(10, 6))
# for i in range(dihedral_angles.shape[1]):
#     plt.plot(dihedral_angles[:25000, i], label=f'Dihedral {i+1}', alpha=0.5)
# plt.xlabel('Frame')
# plt.ylabel('Dihedral Angle (radians)')
# plt.title('Dihedral Angles Over Time')
# plt.legend()
# plt.tight_layout()

# # # Display both visualizations
# # display(view)

In [ ]:
# Check the shape of the data
print(np.array(ref_traj).shape)  # (4, 12500, n_dihedrals)
# print(tica_concatenated.shape) # (62500, 7)
# tica_traj = tica.transform(dihedral_data)
# print(np.array(tica_traj).shape) # (5, 12500, 7)
# print(np.array(cluster.transform(tica_traj)).shape)  # (5, 12500, 1)
# # discretized_traj = tica.discretize(dihedral_data, n_clusters=1000)

In [ ]:
# It's good to scan a wide range of lag times to find the best lag time.
# We want small number of dimensions can explain most of the kinetic variance.
# We should choose the lag time when we see elbow point in that trace.
# QM9 has very few dihedral angles, this might be helpful for drug
def score_cv(data, dim, lag, number_of_splits=4, validation_fraction=0.5):
    """Compute a cross-validated VAMP2 score.

    We randomly split the list of independent trajectories into
    a training and a validation set, compute the VAMP2 score,
    and repeat this process several times.

    Parameters
    ----------
    data : list of numpy.ndarrays
        The input data.
    dim : int
        Number of processes to score; equivalent to the dimension
        after projecting the data with VAMP2.
    lag : int
        Lag time for the VAMP2 scoring.
    number_of_splits : int, optional, default=10
        How often do we repeat the splitting and score calculation.
    validation_fraction : int, optional, default=0.5
        Fraction of trajectories which should go into the validation
        set during a split.
    """
    # we temporarily suppress very short-lived progress bars
    with pyemma.util.contexts.settings(show_progress_bars=False):
        nval = int(len(data) * validation_fraction)
        scores = np.zeros(number_of_splits)
        for n in range(number_of_splits):
            ival = np.random.choice(len(data), size=nval, replace=False)
            vamp = pyemma.coordinates.vamp(
                [d for i, d in enumerate(data) if i not in ival], lag=lag, dim=dim)
            scores[n] = vamp.score([d for i, d in enumerate(data) if i in ival])
    return scores
lags = [1, 3, 10, 30, 100, 300]
dims = [i + 1 for i in range(num_dihedrals)]

fig, ax = plt.subplots()
for i, lag in enumerate(lags):
    scores_ = np.array([score_cv(ref_traj, dim, lag)
                        for dim in dims])
    scores = np.mean(scores_, axis=1)
    errors = np.std(scores_, axis=1, ddof=1)
    color = 'C{}'.format(i)
    ax.fill_between(dims, scores - errors, scores + errors, alpha=0.3, facecolor=color)
    ax.plot(dims, scores, '--o', color=color, label='lag={:.1f}ps'.format(lag * 0.4))
ax.legend()
ax.set_xlabel('number of dimensions')
ax.set_ylabel('VAMP2 score')
fig.tight_layout()

In [ ]:
########## TICA on dihedral angles ##########
# Please play with the lag time and see how it affects the TICA output.
def tica_on_ref(ref_traj, lag=10, plot=False):
    """Compute TICA on the dihedral angles.

    Parameters
    ----------
    ref_traj : list of numpy.ndarrays
        The input data.
    lag : int, optional, default=10
        Lag step for the TICA calculation.
    """
    tica = pyemma.coordinates.tica(ref_traj, lag=lag)
    tica_output = tica.get_output()
    tica_concatenated = np.concatenate(tica_output)
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(10, 3))
        pyemma.plots.plot_feature_histograms(
            tica_concatenated,
            ['IC {}'.format(i + 1) for i in range(tica.dimension())],
            ax=axes[0])
        axes[0].set_title('lag time = {} steps'.format(lag))
        axes[1].set_title(
            'Density, actual dimension = {}'.format(tica.dimension()))
        pyemma.plots.plot_density(
            *tica_concatenated[:, :2].T, ax=axes[1], cbar=False)
        pyemma.plots.plot_free_energy(
            *tica_concatenated[:, :2].T, ax=axes[2], legacy=False)
        for ax in axes[1:].flat:
            ax.set_xlabel('IC 1')
            ax.set_ylabel('IC 2')
        axes[2].set_title('Pseudo free energy')
        fig.tight_layout()

    return tica, tica_output, tica_concatenated

def tica_projection(generated_traj, tica, tica_concatenated, plot=False, num_points_to_plot = 100):
    """Project the generated trajectory onto the TICA space of reference.

    Parameters
    ----------
    generated_traj : list of numpy.ndarrays
        The input data.
    tica : pyemma.coordinates.tica.TICA
        The TICA object fitted on the reference data.
    """
    tica_generated = tica.transform(generated_traj)
    print(*tica_concatenated[:, :2].T.shape)
    if plot:
        print('!!!!!!!!!!!!!!!!!!!!!!! Please be aware that two plots does not have the same range nor color bar!!!!!!!!!!!!!!!!!!!!!!!!')
        fig, axes = plt.subplots(1, 3, figsize=(15, 3))
        pyemma.plots.plot_free_energy(
            *tica_concatenated[:, :2].T, ax=axes[0], legacy=False)
        pyemma.plots.plot_free_energy(
            *tica_generated[:, :2].T, ax=axes[1], legacy=False)

        # Plot the traj projection on the FES, downsampled to num_points 
        pyemma.plots.plot_free_energy(
            *tica_concatenated[:, :2].T, ax=axes[2], legacy=False)
        axes[2].scatter(tica_generated[::tica_generated.shape[0]//num_points_to_plot, 0], 
                tica_generated[::tica_generated.shape[0]//num_points_to_plot, 1], 
                color='black', label='Points', alpha=0.5, s=15) 
        axes[2].plot(tica_generated[::tica_generated.shape[0]//num_points_to_plot, 0], tica_generated[::tica_generated.shape[0]//num_points_to_plot, 1], color='black', label='Connections', alpha=0.5)
        
        axes[0].set_title('Reference FES')
        axes[1].set_title('Generated_traj FES')
        axes[2].set_title('Projection of generated_traj on reference FES')
        
        for ax in axes.flat:
            ax.set_xlabel('IC 1')
            ax.set_ylabel('IC 2')
            # no ticks
            ax.set_xticks([])
            ax.set_yticks([])
    return tica_generated

def kmeans(tica, tica_concatenated, k=100, plot=False):
    """Run kmeans clustering on the TICA output.

    Parameters
    ----------
    tica : pyemma.coordinates.tica.TICA
        The TICA object fitted on the reference data.
    """
    # Run kmeans clustering on the TICA output.
    cluster = pyemma.coordinates.cluster_kmeans(tica, k=k, max_iter=100)
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    pyemma.plots.plot_feature_histograms(
        tica_concatenated, ['IC {}'.format(i + 1) for i in range(tica.dimension())], ax=axes[0])
    pyemma.plots.plot_density(*tica_concatenated[:, :2].T, ax=axes[1], cbar=False, alpha=0.1, logscale=True)
    axes[1].scatter(*cluster.clustercenters[:, :2].T, s=15, c='C1')
    axes[1].set_xlabel('IC 1')
    axes[1].set_ylabel('IC 2')
    axes[1].set_xticks([])
    axes[1].set_yticks([])
    fig.tight_layout()
    axes[1].set_title('Cluster centers on ref FES')
    return cluster, k

In [ ]:
tica, tica_output, tica_concatenated = tica_on_ref(ref_traj, lag=10, plot=True)
tica_generated = tica_projection(generated_traj, tica, tica_concatenated, plot=True)

In [ ]:
cluster,k = kmeans(tica, tica_concatenated, plot=True)
generated_cluster = cluster.transform(tica_generated)

In [ ]:
# MSM analysis, choose lag time first
its = pyemma.msm.its(cluster.dtrajs, lags=[1,2,3,5,10,15,20,50], nits=15, errors='bayes')
pyemma.plots.plot_implied_timescales(its, ylog=True)

In [ ]:
msm = pyemma.msm.estimate_markov_model(cluster.dtrajs, lag=10)  # Put lag time here
msm.pi # stationary distribution of reference
# msm.mfpt(0,1)
ref_metastate_prob = np.zeros(nstates)

In [ ]:
def msm_analysis(cluster, generated_cluster, lag=10, nstates=10):
    """Perform MSM analysis on the clustered data.

    Parameters
    ----------
    cluster : pyemma.coordinates.clustering.Cluster
        The clustering object fitted on the reference data.
    generated_cluster : pyemma.coordinates.clustering.Cluster
        The clustering object fitted on the generated data.
    lag : int, optional, default=10
        Lag step for the MSM calculation.
    nstates : int, optional, default=10
        Number of metastable states to consider.
    """
    # Estimate the MSM model from the clustered data
    msm = pyemma.msm.estimate_markov_model(cluster.dtrajs, lag=lag)
    pcca = msm.pcca(nstates)
    
    microstate_count = np.bincount(generated_cluster.flatten(),minlength=k) # count the number of occurrences of each microstate in the generated trajectory
    microstate_count = np.array(microstate_count/microstate_count.sum()) # use original microstate idx
    # print(microstate_count)

    generated_metastate_prob = np.zeros(nstates) # create buffer for reference metastate probabilities
    for state in range(nstates):
        microstate_idx = msm.active_set[msm.metastable_sets[state]]  # metastable sets use transformed microstates idx. active_set uses original microstates idx.
        # print(microstate_idx) # Now the microstate_idx is in the original space
        generated_metastate_prob[state] = np.sum(microstate_count[microstate_idx])
    print(f'generated metastate prob: {generated_metastate_prob}')

    ref_metastate_prob = np.zeros(nstates)  # create buffer for reference metastate probabilities
    for state in range(nstates):
        microstate_idx_in_state = msm.metastable_sets[state]
        ref_metastate_prob[state] = np.sum(msm.pi[microstate_idx_in_state])
    print(f'ref metastate prob: {ref_metastate_prob}')

    cross_entropy = round(-np.sum(ref_metastate_prob * np.log(generated_metastate_prob + 1e-10)),4) # add small value to avoid log(0)
    print(f'cross entropy: {cross_entropy}')
    
    return msm, cross_entropy

In [ ]:
msm_analysis(cluster, generated_cluster)

In [ ]:
msm = pyemma.msm.estimate_markov_model(cluster.dtrajs, lag=10)  # Put lag time here

print('fraction of states used = {:f}'.format(msm.active_state_fraction))
print('fraction of counts used = {:f}'.format(msm.active_count_fraction))
print(msm.active_set)
# msm.pi.shape

nstates = 10
pcca = msm.pcca(nstates)

# count = np.zeros(nstates)

microstate_count = np.bincount(generated_cluster.flatten(),minlength=k) # count the number of occurrences of each microstate in the generated trajectory
microstate_count = np.array(microstate_count/microstate_count.sum()) # all microstates are included
# print(microstate_count)

generated_metastate_prob = np.zeros(nstates) # create buffer for reference metastate probabilities
for state in range(nstates):
    microstate_idx = msm.active_set[msm.metastable_sets[state]]  # metastable sets use transformed microstates idx. active_set uses original microstates idx.
    # print(microstate_idx) # Now the microstate_idx is in the original space
    generated_metastate_prob[state] = np.sum(microstate_count[microstate_idx])
print(f'generated metastate prob: {generated_metastate_prob}')

ref_metastate_prob = np.zeros(nstates)  # create buffer for reference metastate probabilities
for state in range(nstates):
    microstate_idx_in_state = msm.metastable_sets[state]
    ref_metastate_prob[state] = np.sum(msm.pi[microstate_idx_in_state])
print(f'ref metastate prob: {ref_metastate_prob}')


In [ ]:
msm.pcca(10)

msm.active_set

In [ ]:
msm.pi.shape

In [ ]:
msm.metastable_sets

In [ ]:
msm.mfpt(msm.metastable_sets[0],msm.metastable_sets[-1]) # mean first passage time from state 0 to state 1

In [ ]:
mfpt(A, B): microstate, transformed index
Mean first passage times from set A to set B, in units of the input trajectory time step

Parameters
A (int or int array) – set of starting states

B (int or int array) – set of target states

In [ ]:
msm = pyemma.msm.estimate_markov_model(cluster.dtrajs, lag=10)  # Put lag time here

print('fraction of states used = {:f}'.format(msm.active_state_fraction))
print('fraction of counts used = {:f}'.format(msm.active_count_fraction))
print(msm.active_set)
# Choose 10 metastable states, and assign each cluster to a metastable state.
nstates = 10
pcca = msm.pcca(nstates)
dtrajs_concatenated = np.concatenate(cluster.dtrajs)
metastable_traj = msm.metastable_assignments[dtrajs_concatenated]

# Visualize the metastable states in the TICA space.
fig, ax = plt.subplots(figsize=(5, 4))
_, _, misc = pyemma.plots.plot_state_map(
    *tica_concatenated[:, :2].T, metastable_traj, ax=ax)
ax.set_xlabel('IC 1')
ax.set_ylabel('IC 2')
misc['cbar'].set_ticklabels([r'$\mathcal{S}_%d$' % (i + 1)
                             for i in range(nstates)])
fig.tight_layout()


In [ ]:
# Check whether it's Markovian. nstates = 1 + number of lines which is above the shadow in the plot above
nstates = 3
pyemma.plots.plot_cktest(msm.cktest(nstates));

In [ ]:
metastable_states = msm.metastable_assignments[cluster.transform(tica_traj)]
for i in range(len(metastable_states)):
    traj = metastable_states[i]
    if i == 0:
        counts = np.bincount(traj.flatten(), minlength=10)
        prob_ref = counts / counts.sum()
    # count the number of occurrences of each state
    # in the trajectory and normalize to get the probability distribution
    else:
        counts = np.bincount(traj.flatten(), minlength=10)
        prob = counts / counts.sum()
        # prob = (i == np.arange(10)[:,None]).mean(1)
        print(prob)
        plt.scatter(prob_ref,prob)
x = np.linspace(0, 1, 10)
y = x
plt.plot(x, y, label='x=y', color='gray', linestyle='--')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()


In [ ]:
###### Markov state model stuff #################
if not args.no_msm:
    kmeans, ref_kmeans = mdgen.analysis.get_kmeans(tica.transform(ref))
    try:
        msm, pcca, cmsm = mdgen.analysis.get_msm(ref_kmeans, nstates=10)

        out['kmeans'] = kmeans
        out['msm'] = msm
        out['pcca'] = pcca
        out['cmsm'] = cmsm
    
        traj_discrete = mdgen.analysis.discretize(tica.transform(traj), kmeans, msm)
        ref_discrete = mdgen.analysis.discretize(tica.transform(ref), kmeans, msm)
        out['traj_metastable_probs'] = (traj_discrete == np.arange(10)[:,None]).mean(1)
        out['ref_metastable_probs'] = (ref_discrete == np.arange(10)[:,None]).mean(1)
        ######### 
    
        msm_transition_matrix = np.eye(10)
        for a, i in enumerate(cmsm.active_set):
            for b, j in enumerate(cmsm.active_set):
                msm_transition_matrix[i,j] = cmsm.transition_matrix[a,b]

        out['msm_transition_matrix'] = msm_transition_matrix
        out['pcca_pi'] = pcca._pi_coarse
    
        msm_pi = np.zeros(10)
        msm_pi[cmsm.active_set] = cmsm.pi
        out['msm_pi'] = msm_pi
        
        if args.no_traj_msm:
            pass
        else:
            
            traj_msm = pyemma.msm.estimate_markov_model(traj_discrete, lag=args.msm_lag)
            out['traj_msm'] = traj_msm
    
            traj_transition_matrix = np.eye(10)
            for a, i in enumerate(traj_msm.active_set):
                for b, j in enumerate(traj_msm.active_set):
                    traj_transition_matrix[i,j] = traj_msm.transition_matrix[a,b]
            out['traj_transition_matrix'] = traj_transition_matrix
            
        
            traj_pi = np.zeros(10)
            traj_pi[traj_msm.active_set] = traj_msm.pi
            out['traj_pi'] = traj_pi
            
    except Exception as e:
        print('ERROR', e, name, flush=True)

if args.plot:
    fig.savefig(f'{args.pdbdir}/{name}.pdf')

return name, out

In [ ]:
# fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
# pyemma.plots.plot_contour(
#     *tica_concatenated[:, :2].T,
#     msm.pi[dtrajs_concatenated],
#     ax=axes[0],
#     mask=True,
#     cbar_label='stationary distribution')
# pyemma.plots.plot_free_energy(
#     *tica_concatenated[:, :2].T,
#     weights=np.concatenate(msm.trajectory_weights()),
#     ax=axes[1],
#     legacy=False)
# for ax in axes.flat:
#     ax.set_xlabel('IC 1')
# axes[0].set_ylabel('IC 2')
# axes[0].set_title('Stationary distribution', fontweight='bold')
# axes[1].set_title('Reweighted free energy surface', fontweight='bold')
# fig.tight_layout()